In [1]:
pip install ultralytics huggingface_hub opencv-python

  Using cached matplotlib-3.10.7-cp312-cp312-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (112 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.7 MB/s  0:00:00 eta 0:00:01
Using cached matplotlib-3.10.7-cp312-cp312-macosx_11_0_arm64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.60.1-cp312-cp312-macosx_10_13_universal2.whl (2.8 MB)
Using cached kiwisolver-1.4.9-cp312-cp312-macosx_11_0_arm64.whl (64 kB)
Using cached pyparsing-3.2.5-py3-none-any.whl (113 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import cv2
from pathlib import Path
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, Colors

# --- Configuration ---
# Choisissez le modèle à utiliser : 0: nano, 1: small, 2: medium
# Le modèle nano est le plus rapide, le medium est le plus précis.
MODEL_INDEX = 0 
MODEL_FILES = [
    "yolo11n_doc_layout.pt",
    "yolo11s_doc_layout.pt",
    "yolo11m_doc_layout.pt",
]
SELECTED_MODEL_FILE = MODEL_FILES[MODEL_INDEX]

# Chemin vers votre image de document
IMAGE_PATH = "../images/doc1.png" # <--- MODIFIEZ CECI

# --- Téléchargement du modèle ---
# Crée un dossier pour stocker les modèles téléchargés
DOWNLOAD_PATH = Path("./models")
DOWNLOAD_PATH.mkdir(exist_ok=True)

# Télécharge le modèle depuis le Hub Hugging Face s'il n'est pas déjà présent
try:
    model_path = hf_hub_download(
        repo_id="Armaggheddon/yolo11-document-layout",
        filename=SELECTED_MODEL_FILE,
        repo_type="model",
        local_dir=DOWNLOAD_PATH,
    )
    print(f"Modèle téléchargé à l'emplacement : {model_path}")
except Exception as e:
    print(f"Erreur lors du téléchargement du modèle : {e}")
    exit()


WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/Users/mohammedlbakali/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


yolo11n_doc_layout.pt:   0%|          | 0.00/5.63M [00:00<?, ?B/s]

Modèle téléchargé à l'emplacement : models/yolo11n_doc_layout.pt


In [3]:

# --- Inférence ---
# Charge le modèle YOLO
model = YOLO(model_path)

# Charge l'image
img = cv2.imread(IMAGE_PATH)
if img is None:
    print(f"Erreur : Impossible de charger l'image depuis {IMAGE_PATH}")
    exit()

# Effectue la prédiction
results = model.predict(img)

# --- Visualisation des résultats ---
# Crée un objet pour annoter l'image
annotator = Annotator(img, line_width=2, font_size=8)
colors = Colors() # Pour avoir des couleurs distinctes par classe



0: 1280x928 4 List-items, 1 Page-header, 4 Pictures, 14 Section-headers, 1 Table, 10 Texts, 164.3ms
Speed: 6.9ms preprocess, 164.3ms inference, 7.2ms postprocess per image at shape (1, 3, 1280, 928)


In [ ]:

# Parcours les détections
for result in results:
    boxes = result.boxes
    for box in boxes:
        # Récupère les coordonnées de la boîte englobante
        b = box.xyxy[0]  # format (x1, y1, x2, y2)
        # Récupère la classe (type d'élément) et la confiance
        c = box.cls
        conf = box.conf.item()
        label = f"{model.names[int(c)]} {conf:.2f}"
        
        # Dessine la boîte et le label sur l'image
        annotator.box_label(b, label, color=colors(c, True))

# Récupère l'image annotée
annotated_img = annotator.result()

# Affiche l'image
cv2.imshow("Analyse de la mise en page", annotated_img)
cv2.waitKey(0) # Attend qu'une touche soit pressée pour fermer
cv2.destroyAllWindows()

# Sauvegarde l'image avec les détections
output_path = "resultat_analyse.png"
cv2.imwrite(output_path, annotated_img)
print(f"Image sauvegardée à l'emplacement : {output_path}")